In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../data/processed/final_financial_data.csv")

df["Transaction_Date"] = pd.to_datetime(
    df["Transaction_Date"],
    format="mixed",
    dayfirst=True
)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (9802, 25)


,Transaction_ID,Transaction_Date,Company_Code,Cost_Center,Profit_Center,Account_Category,Transaction_Type,Amount,Budget,Payment_Status,...,Vendor_ID,Tax_Amount,Profit,Approval_Status,Anomaly_Indicator,Year,Month,Quarter,Budget_Variance,Budget_Utilization
0,TX005450,2024-01-17,C003,CC107,PC206,Procurement,Expense,660134.88,870600.30,Pending,...,NaN,115121.44,NaN,Approved,0,2024,1,1,-210465.42,75.825253
1,TX000061,2021-01-21,C001,CC111,PC202,Marketing,Expense,226317.84,255507.62,Paid,...,V040,42898.57,NaN,Approved,0,2021,1,1,-29189.78,88.575769
2,TX007730,2025-10-02,C002,CC105,PC206,Marketing,Expense,381300.39,427732.69,Partially Paid,...,V035,70826.38,NaN,Approved,0,2025,10,4,-46432.30,89.144552
3,TX003695,2023-02-14,C002,CC112,PC204,Procurement,Expense,352712.21,360120.03,Partially Paid,...,V001,62394.66,NaN,Approved,0,2023,2,1,-7407.82,97.942958
4,TX001800,2022-02-25,C005,CC104,PC206,Sales,Revenue,599713.69,603576.07,Paid,...,NaN,109719.59,164961.87,Pending,0,2022,2,1,-3862.38,99.360084


In [4]:
df["Year_Month"] = df["Transaction_Date"].dt.to_period("M")

monthly_summary = (
    df.groupby(["Year_Month", "Transaction_Type"])
      .agg(
          Total_Amount=("Amount", "sum"),
          Total_Budget=("Budget", "sum"),
          Transaction_Count=("Transaction_ID", "count"),
          Total_Tax=("Tax_Amount", "sum")
      )
      .reset_index()
)

monthly_summary.head(10)

,Year_Month,Transaction_Type,Total_Amount,Total_Budget,Transaction_Count,Total_Tax
0,2021-01,Expense,14709639.29,13195904.01,101,1778079.77
1,2021-01,Revenue,8437831.52,8706017.34,17,1501784.11
2,2021-02,Expense,14371577.45,12701169.90,98,1596088.60
3,2021-02,Revenue,11672693.16,12231004.49,26,1986934.15
4,2021-03,Expense,29949213.44,25073185.06,135,3165005.93
5,2021-03,Revenue,15201713.77,14476574.62,25,2568532.97
6,2021-04,Expense,17251021.57,17095298.76,115,1808196.65
7,2021-04,Revenue,11762185.49,10997565.48,20,1843308.76
8,2021-05,Expense,18282795.56,19098853.79,111,2600532.36
9,2021-05,Revenue,9830046.88,10546756.55,20,1765792.97


In [7]:
department_summary = (
    df[df["Transaction_Type"] == "Expense"]
    .groupby("Department")
    .agg(
        Total_Expense=("Amount", "sum"),
        Total_Budget=("Budget", "sum"),
        Transaction_Count=("Transaction_ID", "count"),
        Average_Expense=("Amount", "mean")
    )
    .reset_index()
    .sort_values("Total_Expense", ascending=False)
)

department_summary

,Department,Total_Expense,Total_Budget,Transaction_Count,Average_Expense
5,Operations,3.475406e+08,3.365838e+08,1213,286513.302284
3,IT,3.184716e+08,3.239327e+08,1205,264291.790656
4,Marketing,2.654979e+08,2.517310e+08,1198,221617.601219
7,Sales,2.326824e+08,2.286349e+08,1212,191982.190858
1,Finance,1.995288e+08,1.967396e+08,1221,163414.247797
6,Procurement,1.585050e+08,1.541392e+08,553,286627.494720
2,HR,6.328959e+07,6.082246e+07,549,115281.587869
0,Administration,5.895076e+07,5.636303e+07,1104,53397.426612


In [8]:
budget_summary = (
    df[df["Transaction_Type"] == "Expense"]
    .groupby("Year")
    .agg(
        Actual_Expense=("Amount", "sum"),
        Budget=("Budget", "sum"),
        Transaction_Count=("Transaction_ID", "count")
    )
    .reset_index()
)

budget_summary["Budget_Variance"] = (
    budget_summary["Actual_Expense"] -
    budget_summary["Budget"]
)

budget_summary["Budget_Utilization"] = (
    budget_summary["Actual_Expense"] /
    budget_summary["Budget"]
) * 100

budget_summary

,Year,Actual_Expense,Budget,Transaction_Count,Budget_Variance,Budget_Utilization
0,2021,2.449176e+08,2.383578e+08,1415,6559811.58,102.752086
1,2022,2.792820e+08,2.792917e+08,1524,-9618.66,99.996556
2,2023,3.216218e+08,3.168137e+08,1660,4808109.39,101.517646
3,2024,3.694995e+08,3.649128e+08,1767,4586755.85,101.256946
4,2025,4.291458e+08,4.095709e+08,1889,19574886.68,104.779365


In [10]:
anomaly_results = pd.read_csv(
    "../data/processed/anomaly_detection_results.csv"
)

anomaly_summary = (
    anomaly_results.groupby("Final_Anomaly")
    .agg(
        Transaction_Count=("Transaction_ID", "count"),
        Total_Amount=("Amount", "sum"),
        Average_Amount=("Amount", "mean")
    )
    .reset_index()
)

anomaly_summary["Anomaly_Type"] = anomaly_summary["Final_Anomaly"].map({
    0: "Normal",
    1: "Potential Anomaly"
})

anomaly_summary

,Final_Anomaly,Transaction_Count,Total_Amount,Average_Amount,Anomaly_Type
0,0,9605,2.282428e+09,2.376291e+05,Normal
1,1,197,3.554223e+08,1.804174e+06,Potential Anomaly


In [11]:
forecast_2026 = pd.read_csv(
    "../data/processed/expense_forecast_2026.csv"
)

forecast_2026

,Year_Month,Forecasted_Expense
0,2026-01,33509743.00
1,2026-02,36559223.98
2,2026-03,41634461.56
3,2026-04,41013772.71
4,2026-05,38531266.14
5,2026-06,43197474.22
6,2026-07,39426091.25
7,2026-08,39426091.25
8,2026-09,43185923.86
9,2026-10,39389597.03


In [12]:
import os

os.makedirs("../dashboard/data", exist_ok=True)

print("Dashboard data folder ready.")

Dashboard data folder ready.


In [13]:
monthly_summary.to_csv(
    "../dashboard/data/monthly_summary.csv",
    index=False
)

department_summary.to_csv(
    "../dashboard/data/department_summary.csv",
    index=False
)

budget_summary.to_csv(
    "../dashboard/data/budget_summary.csv",
    index=False
)

anomaly_summary.to_csv(
    "../dashboard/data/anomaly_summary.csv",
    index=False
)

forecast_2026.to_csv(
    "../dashboard/data/forecast_2026.csv",
    index=False
)

print("All dashboard datasets saved successfully.")

All dashboard datasets saved successfully.
